# 7a Verify ID Mapping
Verify that the old place_identifier(year) values from `reference_code/8.ipynb` were correctly
mapped to our current place_identifier(year) values.

**Approach**: The old IDs come from the supplementary dataset (`10_supplementary.csv`). We match
old → new using **phone + year** as the join key (phone numbers are unique per place-year).
Then we compare label, city, and state to confirm the match is correct.

Input:
- data/0_raw/reference/10_supplementary.csv (old IDs + identifying info)
- data/1_derived/5_geocode_truck_stops/3_manual_review_ready.csv (our current IDs)

In [1]:
from pathlib import Path
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'source').exists() and (candidate / 'temp').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SUPP_FILE   = PROJECT_ROOT / 'data' / '0_raw' / 'reference' / '10_supplementary.csv'
REVIEW_FILE = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops' / '3_manual_review_ready.csv'

supp   = pd.read_csv(SUPP_FILE, low_memory=False)
review = pd.read_csv(REVIEW_FILE, low_memory=False)

print(f'Supplementary (old dataset): {len(supp):,} rows, {len(supp.columns)} columns')
print(f'Our review data:             {len(review):,} rows, {len(review.columns)} columns')

Supplementary (old dataset): 2,333 rows, 153 columns
Our review data:             38,135 rows, 141 columns


## The 24 old IDs from reference_code/8.ipynb
These are the place_identifier(year) values used in the manual fixes. They refer to places
in the **old** supplementary dataset, not our current dataset.

In [2]:
old_ids = [10, 89, 92, 135, 164, 247, 253, 273, 275, 277, 283, 284, 291,
           340, 359, 364, 389, 430, 439, 443, 460, 477, 485, 499]

# Look up each old ID in the supplementary dataset
old_info = supp[supp['place_identifier(year)'].isin(old_ids)][
    ['place_identifier(year)', 'label', 'phone', 'city', 'state', 'year']
].drop_duplicates('place_identifier(year)').sort_values('place_identifier(year)')

old_info.columns = ['old_id', 'old_label', 'phone', 'old_city', 'old_state', 'year']
print(f'Found {len(old_info)} of {len(old_ids)} old IDs in supplementary dataset')
old_info

Found 24 of 24 old IDs in supplementary dataset


,old_id,old_label,phone,old_city,old_state,year
54,10,Dunn Oil # 2 ( Chevron ),801-975-7188,Salt Lake City,UT,2006
528,89,Petro Stopping Center # 15,928-757-2799,Kingman,AZ,2006
546,92,Pilot Travel,928-764-2416,Lake Havasu City,AZ,2006
804,135,Wilmington Truck Stop Rd N ),310-830-3388,Wilmington,CA,2006
978,164,North American Truck Stop ),909-350-3310,Fontana,CA,2006
1476,247,USA Fuel Stop,530-865-9645,Orland,CA,2006
1512,253,Cross Country Travel Center ( Shell ),530-347-5353,Cottonwood,CA,2006
1585,273,Fast - Stop ( Chevron ),435-458-3816,Plymouth,UT,2006
1592,275,Lowell Archibald & Sons ( 66 ),435-257-5661,Tremonton,UT,2008
1594,277,Wilson Lane Chevron W ),801-392-5010,Ogden,UT,2006


## Step 1: Match old → new via phone + year
Phone numbers are unique identifiers per place-year. We use them as the join key to find
the corresponding row in our current dataset.

In [3]:
results = []

for _, row in old_info.iterrows():
    old_id = int(row['old_id'])
    phone = row['phone']
    year = row['year']

    # Find matching rows in our dataset by phone + year
    match = review[(review['phone'] == phone) & (review['year'] == year)]

    if len(match) == 0:
        results.append({
            'old_id': old_id, 'phone': phone, 'year': year,
            'old_label': row['old_label'], 'old_city': row['old_city'], 'old_state': row['old_state'],
            'new_id': None, 'new_label': None, 'new_city': None, 'new_state': None,
            'n_matches': 0, 'status': 'NOT FOUND'
        })
    else:
        new_row = match.iloc[0]
        results.append({
            'old_id': old_id, 'phone': phone, 'year': int(year),
            'old_label': row['old_label'], 'old_city': row['old_city'], 'old_state': row['old_state'],
            'new_id': int(new_row['place_identifier(year)']),
            'new_label': new_row['label'], 'new_city': new_row['city'], 'new_state': new_row['state'],
            'n_matches': len(match),
            'status': 'OK' if len(match) == 1 else f'MULTI ({len(match)})'
        })

mapping_df = pd.DataFrame(results)
print(f"Status counts:")
print(mapping_df['status'].value_counts().to_string())
mapping_df

Status counts:
status
OK    24


,old_id,phone,year,old_label,old_city,old_state,new_id,new_label,new_city,new_state,n_matches,status
0,10,801-975-7188,2006,Dunn Oil # 2 ( Chevron ),Salt Lake City,UT,3882,Dunn Oil # 2 ( Chevron ),Salt Lake City,UT,1,OK
1,89,928-757-2799,2006,Petro Stopping Center # 15,Kingman,AZ,3961,Petro Stopping Center # 15,Kingman,AZ,1,OK
2,92,928-764-2416,2006,Pilot Travel,Lake Havasu City,AZ,3964,Pilot Travel,Lake Havasu City,AZ,1,OK
3,135,310-830-3388,2006,Wilmington Truck Stop Rd N ),Wilmington,CA,4027,Wilmington Truck Stop Rd N ),Wilmington,CA,1,OK
4,164,909-350-3310,2006,North American Truck Stop ),Fontana,CA,4056,North American Truck Stop ),Fontana,CA,1,OK
5,247,530-865-9645,2006,USA Fuel Stop,Orland,CA,4139,USA Fuel Stop,Orland,CA,1,OK
6,253,530-347-5353,2006,Cross Country Travel Center ( Shell ),Cottonwood,CA,4145,Cross Country Travel Center ( Shell ),Cottonwood,CA,1,OK
7,273,435-458-3816,2006,Fast - Stop ( Chevron ),Plymouth,UT,7827,Fast - Stop ( Chevron ),Plymouth,UT,1,OK
8,275,435-257-5661,2008,Lowell Archibald & Sons ( 66 ),Tremonton,UT,7829,Lowell Archibald & Sons ( 66 ),Tremonton,UT,1,OK
9,277,801-392-5010,2006,Wilson Lane Chevron W ),Ogden,UT,7831,Wilson Lane Chevron W ),Ogden,UT,1,OK


## Step 2: Verify matches — do label, city, and state agree?
Even though phone+year matched, let's confirm the records describe the same place
by comparing the label (business name), city, and state.

In [4]:
matched = mapping_df[mapping_df['new_id'].notna()].copy()

matched['label_exact_match'] = matched['old_label'].str.strip() == matched['new_label'].str.strip()
matched['city_match']  = matched['old_city'].str.strip() == matched['new_city'].str.strip()
matched['state_match'] = matched['old_state'].str.strip() == matched['new_state'].str.strip()

print('=== Verification Summary ===')
print(f"Label exact match:  {matched['label_exact_match'].sum()} / {len(matched)}")
print(f"City match:         {matched['city_match'].sum()} / {len(matched)}")
print(f"State match:        {matched['state_match'].sum()} / {len(matched)}")
print()

# Show side-by-side comparison
verify = matched[['old_id', 'new_id', 'phone', 'year',
                   'old_label', 'new_label', 'label_exact_match',
                   'old_city', 'new_city', 'city_match',
                   'old_state', 'new_state', 'state_match']].copy()
verify

=== Verification Summary ===
Label exact match:  24 / 24
City match:         24 / 24
State match:        24 / 24



,old_id,new_id,phone,year,old_label,new_label,label_exact_match,old_city,new_city,city_match,old_state,new_state,state_match
0,10,3882,801-975-7188,2006,Dunn Oil # 2 ( Chevron ),Dunn Oil # 2 ( Chevron ),True,Salt Lake City,Salt Lake City,True,UT,UT,True
1,89,3961,928-757-2799,2006,Petro Stopping Center # 15,Petro Stopping Center # 15,True,Kingman,Kingman,True,AZ,AZ,True
2,92,3964,928-764-2416,2006,Pilot Travel,Pilot Travel,True,Lake Havasu City,Lake Havasu City,True,AZ,AZ,True
3,135,4027,310-830-3388,2006,Wilmington Truck Stop Rd N ),Wilmington Truck Stop Rd N ),True,Wilmington,Wilmington,True,CA,CA,True
4,164,4056,909-350-3310,2006,North American Truck Stop ),North American Truck Stop ),True,Fontana,Fontana,True,CA,CA,True
5,247,4139,530-865-9645,2006,USA Fuel Stop,USA Fuel Stop,True,Orland,Orland,True,CA,CA,True
6,253,4145,530-347-5353,2006,Cross Country Travel Center ( Shell ),Cross Country Travel Center ( Shell ),True,Cottonwood,Cottonwood,True,CA,CA,True
7,273,7827,435-458-3816,2006,Fast - Stop ( Chevron ),Fast - Stop ( Chevron ),True,Plymouth,Plymouth,True,UT,UT,True
8,275,7829,435-257-5661,2008,Lowell Archibald & Sons ( 66 ),Lowell Archibald & Sons ( 66 ),True,Tremonton,Tremonton,True,UT,UT,True
9,277,7831,801-392-5010,2006,Wilson Lane Chevron W ),Wilson Lane Chevron W ),True,Ogden,Ogden,True,UT,UT,True


## Step 3: Inspect any mismatches
If any label, city, or state didn't match exactly, show them here for manual review.

In [5]:
mismatches = matched[~(matched['label_exact_match'] & matched['city_match'] & matched['state_match'])]

if len(mismatches) == 0:
    print('All 24 matches verified: label, city, and state all agree.')
else:
    print(f'{len(mismatches)} record(s) with at least one field mismatch:')
    for _, r in mismatches.iterrows():
        print(f"\n  old_id={int(r['old_id'])} -> new_id={int(r['new_id'])}  (phone={r['phone']}, year={int(r['year'])})")
        if not r['label_exact_match']:
            print(f"    Label:  OLD='{r['old_label']}'")
            print(f"            NEW='{r['new_label']}'")
        if not r['city_match']:
            print(f"    City:   OLD='{r['old_city']}' vs NEW='{r['new_city']}'")
        if not r['state_match']:
            print(f"    State:  OLD='{r['old_state']}' vs NEW='{r['new_state']}')")

All 24 matches verified: label, city, and state all agree.


## Final mapping table
The confirmed old_id → new_id mapping used by `7_apply_manual_fixes.ipynb`.

In [6]:
final_map = matched[['old_id', 'new_id', 'phone', 'year', 'old_label', 'old_city', 'old_state',
                      'label_exact_match', 'city_match', 'state_match']].copy()
final_map['new_id'] = final_map['new_id'].astype(int)
final_map = final_map.sort_values('old_id').reset_index(drop=True)
print(f'{len(final_map)} mappings total')
final_map

24 mappings total


,old_id,new_id,phone,year,old_label,old_city,old_state,label_exact_match,city_match,state_match
0,10,3882,801-975-7188,2006,Dunn Oil # 2 ( Chevron ),Salt Lake City,UT,True,True,True
1,89,3961,928-757-2799,2006,Petro Stopping Center # 15,Kingman,AZ,True,True,True
2,92,3964,928-764-2416,2006,Pilot Travel,Lake Havasu City,AZ,True,True,True
3,135,4027,310-830-3388,2006,Wilmington Truck Stop Rd N ),Wilmington,CA,True,True,True
4,164,4056,909-350-3310,2006,North American Truck Stop ),Fontana,CA,True,True,True
5,247,4139,530-865-9645,2006,USA Fuel Stop,Orland,CA,True,True,True
6,253,4145,530-347-5353,2006,Cross Country Travel Center ( Shell ),Cottonwood,CA,True,True,True
7,273,7827,435-458-3816,2006,Fast - Stop ( Chevron ),Plymouth,UT,True,True,True
8,275,7829,435-257-5661,2008,Lowell Archibald & Sons ( 66 ),Tremonton,UT,True,True,True
9,277,7831,801-392-5010,2006,Wilson Lane Chevron W ),Ogden,UT,True,True,True
